<h1>LSTM Analysis</h1>

In [24]:
import os
import sys
# Setting Hadoop home directory for the JVM
os.environ["HADOOP_HOME"] = r'C:\hadoop'
os.environ["PATH"] = r'C:\hadoop\bin;' + os.environ["PATH"]

In [25]:
import findspark
findspark.init()
import pyspark
import pandas as pd
import yfinance as yf
from pyspark.sql.functions import col, round, when, lag, avg, first, stddev
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql import SparkSession

jar_path = r"D:\\PostgreSQL\\postgressql\\postgresql-42.7.12.jar"
spark =( SparkSession.builder
        .master("local[*]")
        .config("spark.driver.extraClassPath", jar_path)
        .config("spark.executor.extraClassPath", jar_path)
        .config('spark.hadoop.home.dir', r'C:\hadoop')
        .config('spark.driver.extraJavaOptions', '-Dhadoop.home.dir=C:\\hadoop')
        .appName("LSTM Author Classification")
        .getOrCreate()
)
print('Spark Version:', spark.version)
print('Hadoop Home:', os.environ["HADOOP_HOME"])

Spark Version: 3.5.2
Hadoop Home: C:\hadoop


In [26]:
# Set dataset path
DATASET_PATH = r"D:\DataScience\PostGrad\PDAN02\LSTM\dataset"
# Read text file
raw_df = (spark.read.format('binaryFile')
          .option('recursiveFileLookup','true')
          .option('pathGlobalFilter','*.txt')
          .load(DATASET_PATH)
          .withColumn("content", F.expr("decode(content, 'UTF-8')"))
          .select("path", "content")
)

In [27]:
# Parsing author and book_id from file path
parsed_df =(
    raw_df
    .withColumn('normal_path',F.regexp_replace('path',r'\\','/'))
    .withColumn('author',F.regexp_extract(F.col("normal_path"), r'/dataset/([^/]+)/', 1))
    .withColumn('file_name',F.regexp_extract(F.col("normal_path"), r'/([^/]+)$', 1))
    .withColumn('book_name',F.regexp_replace(F.col("file_name"), r'\.[^.]+$', ""))
    .filter(F.col('author')!="")
    .filter(F.col("content").isNotNull())
    
)

In [28]:
# Setting up Chunk Size - Earch Row will have 200 words
chunk_size = 200
# Split text into words
words_df = (
    parsed_df
    .withColumn("words",F.split(
        F.trim(F.regexp_replace(F.col("content"), r"\s+"," ")), " ")
    )
    .filter(F.size(F.col("words")) >= chunk_size)
)

In [29]:
# Create non-overlapping chunks of 200 words
chunks_df = (
    words_df
    .withColumn("starts",
            F.sequence(
                F.lit(0),
                F.size(F.col("words")) - chunk_size,
                F.lit(chunk_size)
            )
    )
    .withColumn("chunks",
                F.transform(
                    F.col("starts"),
                    lambda s: F.array_join(F.slice(F.col('words'), s + 1, chunk_size), " ")
                )
    )
    .select("author","book_name", F.explode("chunks").alias("text"))
    .filter(F.length(F.trim(F.col("text"))) > 0)
)

In [30]:
# Displaying Spark Dataframe
#chunks_df.printSchema()
#chunks_df.show(5, truncate=120)

In [ ]:
# Count Chunks per Author
chunks_df.groupBy('author').count().orderBy(F.desc('count')).show(truncate=False)

In [ ]:
# Saving Dataframe as CSV
chunks_df.write.mode("overwrite").option('header', True).csv(r"D:\DataScience\PostGrad\PDAN02\LSTM\dataset\author_classification_df.csv")